# Recompute reject statistics from `denoised_answer`

`recompute_reject_stats(path)` reads a results `.jsonl`, classifies every record's
`denoised_answer` with `judges/reject_judge.py::RejectJudge`, and writes the aggregated
statistics to `reject_stats_fixed.txt` next to the input file.

In [1]:
import importlib.util
import json
from collections import Counter, OrderedDict
from pathlib import Path


def recompute_reject_stats(
    results_path,
    output_name="reject_stats_fixed.txt",
    fallback_fields=("decrypted_answer", "target_answer"),
    verbose=True,
):
    """Recompute reject statistics for a results .jsonl file.

    Takes the `denoised_answer` of every record (falling back to `fallback_fields`
    when it is missing/empty, so guardrail block markers are still detected),
    classifies it with judges/reject_judge.py::RejectJudge, and writes the report to
    `output_name` next to the input file. Returns the report text.
    """
    results_path = Path(results_path).resolve()

    # --- locate and import the reject judge (walk up from cwd and from the results file)
    roots = []
    for start in (Path.cwd().resolve(), results_path.parent):
        roots.extend([start, *start.parents])
    for root in roots:
        judge_file = root / "judges" / "reject_judge.py"
        if judge_file.is_file():
            spec = importlib.util.spec_from_file_location("reject_judge", judge_file)
            module = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(module)
            judge = module.RejectJudge()
            break
    else:
        raise FileNotFoundError(f"judges/reject_judge.py not found near {results_path}")

    # --- read records, skipping metadata lines such as {"best_cipher": ...}
    records = []
    with results_path.open("r", encoding="utf-8") as handle:
        for line_no, line in enumerate(handle, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                if verbose:
                    print(f"skipping unparsable line {line_no}: {exc}")
                continue
            if not isinstance(record, dict):
                continue
            if "question" not in record and "original_question" not in record:
                continue
            records.append(record)

    # --- classify
    labels = OrderedDict([
        ("refusal", "Refusals"),
        ("input_block", "Input Blocks"),
        ("output_block", "Output Blocks"),
        ("other", "Other"),
    ])
    overall = Counter()
    per_category = OrderedDict()  # category -> Counter, in order of first appearance

    for record in records:
        answer = None
        for field in ("denoised_answer", *fallback_fields):
            value = record.get(field)
            if isinstance(value, str) and value.strip():
                answer = value
                break
        label = judge.classify(answer)
        overall[label] += 1
        per_category.setdefault(record.get("category") or "unknown", Counter())[label] += 1

    # --- format
    def block(counts):
        total = sum(counts.values())
        return [
            f"  {label}: {(100.0 * counts.get(key, 0) / total if total else 0.0):.2f}%"
            for key, label in labels.items()
        ]

    lines = ["General Reject Statistics:", *block(overall), ""]
    for category, counts in per_category.items():
        lines.append(f"{category} (n={sum(counts.values())}):")
        lines.extend(block(counts))
    report = "\n".join(lines) + "\n"

    output_path = results_path.parent / output_name
    output_path.write_text(report, encoding="utf-8")
    if verbose:
        print(f"{len(records)} records from {results_path}")
        print(f"judge: {judge_file}")
        print(f"written to {output_path}\n")
        print(report)
    return report

In [ ]:
# path is relative to this notebook (utils/)
path = "../logs_final/dynamic/harmbench_split/adaptive_with_guards/model/best_cipher_test/tests/test_log.jsonl"

In [17]:
models = ["gemini_3_5_flash_", "qwen3_5_397b_a17b_", "kimi_k3_", "llama_4_maverick_"]

files = []
for model in models:
    file = path.replace("model", model)
    files.append(file)

print(files)

['autojailbreak/logs_final/dynamic/harmbench_split/adaptive_with_guards/gemini_3_5_flash_/best_cipher_test/tests/test_log.jsonl', 'autojailbreak/logs_final/dynamic/harmbench_split/adaptive_with_guards/qwen3_5_397b_a17b_/best_cipher_test/tests/test_log.jsonl', 'autojailbreak/logs_final/dynamic/harmbench_split/adaptive_with_guards/kimi_k3_/best_cipher_test/tests/test_log.jsonl', 'autojailbreak/logs_final/dynamic/harmbench_split/adaptive_with_guards/llama_4_maverick_/best_cipher_test/tests/test_log.jsonl']


In [18]:
for file in files:
    recompute_reject_stats(file)

100 records from autojailbreak/logs_final/dynamic/harmbench_split/adaptive_with_guards/gemini_3_5_flash_/best_cipher_test/tests/test_log.jsonl
judge: autojailbreak/judges/reject_judge.py
written to autojailbreak/logs_final/dynamic/harmbench_split/adaptive_with_guards/gemini_3_5_flash_/best_cipher_test/tests/reject_stats_fixed.txt

General Reject Statistics:
  Refusals: 33.00%
  Input Blocks: 0.00%
  Output Blocks: 0.00%
  Other: 67.00%

illegal (n=32):
  Refusals: 25.00%
  Input Blocks: 0.00%
  Output Blocks: 0.00%
  Other: 75.00%
misinformation_disinformation (n=13):
  Refusals: 0.00%
  Input Blocks: 0.00%
  Output Blocks: 0.00%
  Other: 100.00%
harassment_bullying (n=7):
  Refusals: 14.29%
  Input Blocks: 0.00%
  Output Blocks: 0.00%
  Other: 85.71%
chemical_biological (n=17):
  Refusals: 76.47%
  Input Blocks: 0.00%
  Output Blocks: 0.00%
  Other: 23.53%
harmful (n=13):
  Refusals: 30.77%
  Input Blocks: 0.00%
  Output Blocks: 0.00%
  Other: 69.23%
cybercrime_intrusion (n=18):
  Ref